In [16]:
import pandas as pd
import pyarrow.dataset as ds
from pathlib import Path

from services.db import TRAINING_DIR

TRAIN_DIR = Path(TRAINING_DIR)

dataset = ds.dataset(TRAIN_DIR.as_posix(), format="parquet", partitioning="hive")
dataset.schema


time_idx: int64
series_ticker: string
status: string
ticker: string
timestamp: timestamp[ns]
bid: int64
ask: int64
spread: int64
volume: int64
bid_count: int64
ask_count: int64
obi: double
spread_velocity: double
momentum_5: double
momentum_10: double
momentum_20: double
date: string
category: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 2163

# What partitions do we have?

In [17]:
# list partitions by scanning the folder structure
dates = sorted({p.name.split("=")[1] for p in TRAIN_DIR.glob("date=*") if p.is_dir()})
dates[:10], dates[-10:], len(dates)


(['2026-02-06', '2026-02-07', '2026-02-08'],
 ['2026-02-06', '2026-02-07', '2026-02-08'],
 3)

# Row count by day

In [18]:
import pyarrow.compute as pc

table = dataset.to_table(columns=["date"])
df_dates = table.to_pandas()
df_dates["date"].value_counts().sort_index()


date
2026-02-06     799214
2026-02-07    1986157
2026-02-08    1172299
Name: count, dtype: int64

In [4]:
cols = ["timestamp","ticker","bid","ask","spread","volume","bid_count","ask_count","obi","date"]
t = dataset.to_table(columns=cols)  # if this is too big, filter by one date first
df = t.to_pandas()

df[["bid","ask","spread","volume","bid_count","ask_count","obi"]].describe()
df["spread"].value_counts().head(20)


spread
1     1851909
2      441386
3      168908
4       94565
5       70545
6       39270
7       33328
8       18743
10      12742
9       11913
12       8127
11       7743
13       6976
15       6546
14       5764
18       4822
17       4312
19       3982
16       3454
27       2425
Name: count, dtype: int64

# distributions (spread, prices, volume)

In [5]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
daily_counts = (df.groupby(["date","ticker"])
                  .size()
                  .rename("rows")
                  .reset_index())

daily_counts["rows"].describe()
daily_counts.sort_values("rows", ascending=False).head(20)


,date,ticker,rows
1087,2026-02-06,KXPGATOUR-WMPO26-MMCG,5838
1025,2026-02-06,KXNFLWINMARGIN-26FEB08SEANE-1114,5345
1105,2026-02-06,KXPGATOUR-WMPO26-XSCH,4850
284,2026-02-06,KXEPLGAME-26FEB12BREARS-ARS,4786
1026,2026-02-06,KXNFLWINMARGIN-26FEB08SEANE-13,4775
1037,2026-02-06,KXNYCSNOWM-26FEB-10.0,4608
1114,2026-02-06,KXPRESNOMD-28-JS,4598
3378,2026-02-07,KXPGATOUR-WMPO26-NTAY,4563
3358,2026-02-07,KXPGATOUR-WMPO26-CMOR,4521
1110,2026-02-06,KXPRESNOMD-28-AB,4479


# Per-ticker bar density

In [6]:
(daily_counts["rows"] >= 200).mean()


np.float64(0.501167976215757)

# compute how many tickers have at least 200 rows/day

In [7]:
dup = (df.groupby(["ticker","timestamp"]).size().rename("n").reset_index())
dup[dup["n"] > 1].head(20)
print("duplicate ticker-timestamp pairs:", (dup["n"] > 1).sum())


duplicate ticker-timestamp pairs: 107414


# Duplicate timestamps per ticker

In [8]:
sample_ticker = daily_counts.sort_values("rows", ascending=False).iloc[0]["ticker"]
s = df[df["ticker"] == sample_ticker].sort_values("timestamp").copy()
s["dt"] = s["timestamp"].diff().dt.total_seconds()

s["dt"].describe()
s["dt"].value_counts().head(10)
s[s["dt"] > 120].head(10)   # gaps > 2 minutes


,timestamp,ticker,bid,ask,spread,volume,bid_count,ask_count,obi,date,dt
661239,2026-02-06 17:52:57.484955788,KXPGATOUR-WMPO26-MMCG,1,99,1,258908,2150,448662,-0.990462,2026-02-06,4251.291820
455188,2026-02-06 22:06:44.993258953,KXPGATOUR-WMPO26-MMCG,0,99,1,262353,0,454575,-1.000000,2026-02-06,14986.301184


# Gaps / continuity check

In [9]:
s["mid"] = (s["bid"] + s["ask"]) / 2.0
h = 20  # 10 minutes ahead at 30s bars
s["mid_fwd"] = s["mid"].shift(-h)
s["ret_fwd"] = (s["mid_fwd"] - s["mid"])  # in cents

s["ret_fwd"].describe()


count    5818.000000
mean       -0.001719
std         0.029268
min        -0.500000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.000000
Name: ret_fwd, dtype: float64

# Create “mid price” and quick forward return target

In [10]:
# If we buy YES at ask, we need mid to rise enough to overcome half spread (rough proxy)
s["edge_needed"] = (s["spread"] / 2.0)
s["win_proxy"] = (s["ret_fwd"] > s["edge_needed"]).astype(int)
s["win_proxy"].mean()


np.float64(0.0)

# Quick “spread-aware win” toy label

In [11]:
# If we buy YES at ask, we need mid to rise enough to overcome half spread (rough proxy)
s["edge_needed"] = (s["spread"] / 2.0)
s["win_proxy"] = (s["ret_fwd"] > s["edge_needed"]).astype(int)
s["win_proxy"].mean()


np.float64(0.0)

In [12]:
import os, glob
import pandas as pd

from services.db import TRAINING_DIR

TRAIN_DIR = Path(TRAINING_DIR).as_posix()

# Load a recent partition (fastest way to start)
latest = max(glob.glob(os.path.join(TRAIN_DIR, "date=*/**/*.parquet"), recursive=True), key=os.path.getmtime)
df = pd.read_parquet(latest)

df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   time_idx         750 non-null    int64         
 1   series_ticker    750 non-null    object        
 2   status           750 non-null    object        
 3   ticker           750 non-null    object        
 4   timestamp        750 non-null    datetime64[ns]
 5   bid              750 non-null    int64         
 6   ask              750 non-null    int64         
 7   spread           750 non-null    int64         
 8   volume           750 non-null    int64         
 9   bid_count        750 non-null    int64         
 10  ask_count        750 non-null    int64         
 11  obi              750 non-null    float64       
 12  spread_velocity  750 non-null    int64         
 13  momentum_5       750 non-null    float64       
 14  momentum_10      750 non-null    float64  

,time_idx,series_ticker,status,ticker,timestamp,bid,ask,spread,volume,bid_count,ask_count,obi,spread_velocity,momentum_5,momentum_10,momentum_20
0,76,unknown,active,KXLALIGAGAME-26FEB08VCFRMA-RMA,2026-02-08 02:30:26.806543827,1,99,1,195597,146686,187657,-0.122542,0,0.0,0.0,0.0
1,46,unknown,active,KXBUNDESLIGAGAME-26FEB08BMUTSG-BMU,2026-02-08 02:30:27.842895031,1,99,2,157914,219418,115474,0.310381,0,0.0,0.0,0.0
2,46,unknown,active,KXLALIGAGAME-26FEB08ATMRBB-ATM,2026-02-08 02:30:28.881015062,1,99,2,144787,141316,212486,-0.201158,0,0.0,0.0,0.0
3,46,unknown,active,KXSPACEXCOUNT-26FEB-10,2026-02-08 02:30:30.252120018,1,99,7,129069,8035,7300,0.047930,0,0.0,0.0,0.0
4,46,unknown,active,KXBTCMINMON-BTC-26FEB28-5750000,2026-02-08 02:30:31.312950134,1,99,1,107460,24550,5745,0.620729,0,0.0,0.0,0.0


# series ticker and category populated?

In [14]:
df["series_ticker"].value_counts(dropna=False).head(20)
df["category"].value_counts(dropna=False).head(20)


KeyError: 'category'

# which tickers have enough rows for training

In [ ]:
# Each row is a snapshot; you resample later, but this is still useful
counts = df.groupby("ticker")["timestamp"].count().sort_values(ascending=False)
counts.head(20), counts.describe()


# spread vs. obi

In [ ]:
df["spread"].describe()
df["obi"].describe()

# How often is spread tight enough to plausibly trade?
(df["spread"] <= 2).mean(), (df["spread"] <= 5).mean()


In [20]:
import pandas as pd
df = pd.read_csv("/Users/jrg/ev/Project_K/artifacts/models/policy/20260208_200328_f30s_h20/trade_logs_test.csv")
print(df.columns.tolist())
print(df.head(20).to_string(index=False))
print("\nRealized pnl quantiles:", df["realized_pnl"].quantile([0.1,0.5,0.9]).to_dict())

['timestamp', 'ticker', 'series_ticker', 'series_id', 'time_idx', 'side', 'pred_long', 'pred_short', 'pred_edge', 'realized_pnl', 'spread', 'staleness_sec', 'has_obs']
          timestamp                             ticker    series_ticker                            series_id  time_idx  side  pred_long  pred_short  pred_edge  realized_pnl  spread  staleness_sec  has_obs
2026-02-08 00:15:00       KXATPMATCH-26FEB08GASOCO-OCO       KXATPMATCH       KXATPMATCH-26FEB08GASOCO-OCO:0         0 SHORT -10.498905    6.538221   6.538221          -2.0     2.0            0.0        1
2026-02-08 00:15:30        KXNBAGAME-26FEB09DETCHA-CHA        KXNBAGAME        KXNBAGAME-26FEB09DETCHA-CHA:0       326 SHORT  -9.679064    5.384743   5.384743          -2.0     2.0            0.0        1
2026-02-08 00:24:00    KXNCAAMBTOTAL-26FEB07TENNUK-147    KXNCAAMBTOTAL    KXNCAAMBTOTAL-26FEB07TENNUK-147:0       118 SHORT  -4.235220    1.012561   1.012561          -1.0     1.0            0.0        1
2026-02-08 0